In [4]:
import pandas as pd

from utils.datapath import edinet_codes_path, adj_share_counts_path
from utils.datetime import date_to_quarter

In [5]:
pd.set_option('display.max_rows', None)

In [6]:
df = pd.read_excel(edinet_codes_path)
df.info()

FileNotFoundError: [Errno 2] No such file or directory: '../refdata/edinet_codes.xlsx'

In [4]:
df.columns

Index(['EDINET Code', 'Type of Submitter', 'Listed company / Unlisted company',
       'Consolidated / NonConsolidated', 'Capital stock',
       'account closing date', 'Submitter Name', 'Submitter Name（alphabetic）',
       'Submitter Name（phonetic）', 'Province', 'Submitter's industry',
       'Securities Identification Code', 'Submitter's Japan Corporate Number'],
      dtype='object')

In [5]:
# Ensure all SIC ends with 0
columns = {
    "EDINET Code": "edinet_code",
    "Submitter Name（alphabetic）": "company_name",
    "Submitter's industry": "industry",
    "Securities Identification Code": "code",
}

valid_company_df = df[columns.keys()].dropna()

last_0 = valid_company_df["Securities Identification Code"].apply(
    lambda x: str(x)[-1] == "0"
)
valid_company_df[~last_0]

,EDINET Code,Submitter Name（alphabetic）,Submitter's industry,Securities Identification Code


In [6]:
valid_company_df = valid_company_df.rename(columns = columns)
valid_company_df["code"] = valid_company_df["code"].apply(lambda x: str(x)[0:-1]) # Remove trailing "0"

In [7]:
# valid_company_df.to_dict('records')
# with open(all_securities_path, 'w', encoding='utf-8') as f:
#     json.dump(valid_company_df.to_dict('records'), f)

In [2]:
from utils.data import all_securities

stock_list = [security["code"] for security in all_securities]

In [3]:
import os
import requests
import warnings

from dotenv import load_dotenv
from tqdm import tqdm

warnings.filterwarnings('ignore')

load_dotenv()
JQUANT_API_KEY = os.getenv("JQUANT_API_KEY")

API_URL = "https://api.jquants.com"

In [4]:
refreshtoken = JQUANT_API_KEY

res = requests.post(f"{API_URL}/v1/token/auth_refresh?refreshtoken={refreshtoken}")
if res.status_code == 200:
    id_token = res.json()['idToken']
    headers = {'Authorization': 'Bearer {}'.format(id_token)}
    display("You got an idToken")
else:
    display(res.json()["message"])


'You got an idToken'

In [5]:
def get_average_share_count_adj_series(code, headers):
    params = {}
    params["code"] = code
    res = requests.get(f"{API_URL}/v1/fins/statements", params=params, headers=headers)
    if res.status_code == 200:
        df = pd.DataFrame(res.json()["statements"])
        
        shares_df = df[["CurrentPeriodEndDate", "AverageNumberOfShares"]]
        shares_df["CurrentPeriodEndDate"] = pd.to_datetime(shares_df["CurrentPeriodEndDate"])
        shares_df = shares_df[shares_df["AverageNumberOfShares"].str.isdigit()].sort_values("CurrentPeriodEndDate")
        
        shares_df["Quarter"] = shares_df["CurrentPeriodEndDate"].apply(date_to_quarter)
        shares_df = shares_df.drop_duplicates(subset="Quarter")
        shares_df = shares_df[["Quarter", "AverageNumberOfShares"]]
        
        shares_df["AverageNumberOfShares"] = shares_df["AverageNumberOfShares"].astype(int)
        shares_df["split_factor"] = shares_df["AverageNumberOfShares"].pct_change().round(decimals=3) + 1
        shares_df["split_factor"] = shares_df["split_factor"].shift(-1)
        shares_df["split_factor"] = shares_df["split_factor"].iloc[::-1].cumprod().iloc[::-1].ffill()

        shares_df["AverageNumberOfSharesAdj"] = (shares_df["AverageNumberOfShares"] * shares_df["split_factor"]).astype(int)
        
        return shares_df.set_index("Quarter")["AverageNumberOfSharesAdj"].rename(code)
    else:
        return pd.Series(name=code)

In [8]:
complete_shares_df = pd.DataFrame()
error_codes = []

for code in tqdm(stock_list):
    try:
        series = get_average_share_count_adj_series(code, headers)
        complete_shares_df = pd.concat([complete_shares_df, series.to_frame()], axis=1)
    except:
        print(f"Error for stock code: {code}")
        error_codes.append(code)

# Error for stock code: 187A
# Error for stock code: 9388

  0%|          | 0/3578 [00:00<?, ?it/s]

 11%|█▏        | 410/3578 [03:27<23:40,  2.23it/s]

Error for stock code: 187A


 98%|█████████▊| 3506/3578 [29:35<00:32,  2.19it/s]

Error for stock code: 9388


100%|██████████| 3578/3578 [30:08<00:00,  1.98it/s]


In [ ]:
complete_shares_df = complete_shares_df.sort_index()
# complete_shares_df.to_csv(adj_share_counts_path)

In [13]:
complete_shares_df

,6178,8316,8411,8306,9501,9432,8421,6758,4612,7203,...,4894,9330,5889,151A,135A,143A,9345,3045,5591,3350
Quarter,,,,,,,,,,,,,,,,,,,,,
2014-Q1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-Q4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-Q1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-Q2,NaN,3.903605e+09,2.528081e+09,1.164707e+10,1.602365e+09,8.343758e+10,8700240.0,6.052410e+09,2.345976e+09,1.319752e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2146047.0,NaN,4.075926e+08
2015-Q3,3.103619e+09,3.903595e+09,2.529171e+09,1.165004e+10,1.602359e+09,8.344884e+10,8703028.0,6.050962e+09,2.345973e+09,1.319822e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2146717.0,NaN,4.074673e+08
2015-Q4,3.102263e+09,3.903585e+09,2.529108e+09,1.164881e+10,1.602354e+09,8.344720e+10,8703084.0,6.050386e+09,2.345971e+09,1.320082e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2147521.0,NaN,4.074818e+08
2016-Q1,3.100761e+09,3.903578e+09,2.528011e+09,1.164498e+10,1.602347e+09,8.348796e+10,8704061.0,6.052763e+09,2.345970e+09,1.320621e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2148330.0,NaN,4.074818e+08


In [ ]:
# Create baskets
# Top 1000 stocks by market cap per quarter

# TODO:
# 1. Create function to get historical market cap for single stock
# 3. Get historical shares outstanding - Done (for last 10 years)
# 3. Adjust for stock splits - Done

# 4. Compute market cap by multiplying with prices
# 5. Form quarterly baskets
# 6. Form quarterly baskets with current market cap (to see difference) 
